#  Week 5, Day 4 — June 18, 2026


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import seaborn as sns, os, json, warnings
warnings.filterwarnings('ignore')
BASE_DIR = "/content/drive/MyDrive/Ocean_Plastic_Project"
DIRS = {"raw":BASE_DIR+"/data/raw","processed":BASE_DIR+"/data/processed",
        "metadata":BASE_DIR+"/data/metadata","visualizations":BASE_DIR+"/visualizations"}
print("Environment ready ")
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [ ]:
scaled_df = pd.read_csv(DIRS['processed']+'/cluster_features_scaled.csv')
X_scaled  = scaled_df[['SST_scaled','pH_scaled','Species_scaled']].values
FEATURES  = ['SST (°C)','pH Level','Species Observed']
print(f'Feature matrix loaded: {X_scaled.shape}')

## Step 1: Fit K-Means

In [ ]:
km = KMeans(n_clusters=3, random_state=42, n_init=10, max_iter=300, algorithm='lloyd')
km.fit(X_scaled)
labels = km.labels_; inertia = km.inertia_; n_iter = km.n_iter_

print('K-MEANS TRAINING COMPLETE')
print('='*45)
print(f'  Final inertia : {inertia:.4f}')
print(f'  Iterations    : {n_iter}')
print()
print('  Cluster sizes:')
for c in range(3):
    n = (labels==c).sum(); pct = n/len(labels)*100
    print(f'    Cluster {c}: {n:3d} records ({pct:.1f}%)')

## Step 2: Profile Clusters & Assign Risk Labels

In [ ]:
cluster_df = scaled_df[FEATURES].copy()
cluster_df['Cluster_Raw'] = labels
profiles   = cluster_df.groupby('Cluster_Raw')[FEATURES].mean()
counts     = cluster_df['Cluster_Raw'].value_counts().sort_index()

print('CLUSTER PROFILES (original scale):')
print('='*65)
print(f'  {"Cluster":<10} {"SST (°C)":>10} {"pH Level":>10} {"Species":>12}  n')
print('-'*65)
for c in range(3):
    row = profiles.loc[c]
    print(f'  Cluster {c}    {row["SST (°C)"]:>10.3f} {row["pH Level"]:>10.5f} '
          f'{row["Species Observed"]:>12.2f}  {counts[c]}')
print()

# Assign risk: lowest SST = Stable, highest SST = Critical, middle = At-Risk
risk_map = {}
for c in range(3):
    sst = profiles.loc[c,'SST (°C)']
    if   sst == profiles['SST (°C)'].max(): risk_map[c] = 'Critical'
    elif sst == profiles['SST (°C)'].min(): risk_map[c] = 'Stable'
    else:                                    risk_map[c] = 'At_Risk'

print('RISK LABEL ASSIGNMENT (by SST rank):')
for c,label in risk_map.items():
    row = profiles.loc[c]
    print(f'  Cluster {c} → {label:<10} (SST={row["SST (°C)"]:.3f}°C, Species={row["Species Observed"]:.2f})')

cluster_df['Risk_Cluster'] = cluster_df['Cluster_Raw'].map(risk_map)
print()
print('Final Risk Cluster distribution:')
print(cluster_df['Risk_Cluster'].value_counts())

## Step 3: Cluster Scatter Plots (3 Feature Pair Views)

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(18,6))
sns.set_style('whitegrid')
risk_colors = {'Stable':'#27ae60','At_Risk':'#f39c12','Critical':'#e74c3c'}

for ax,(xf,yf) in zip(axes,[('SST (°C)','Species Observed'),
                              ('pH Level','Species Observed'),
                              ('SST (°C)','pH Level')]):
    for risk,color in risk_colors.items():
        sub = cluster_df[cluster_df['Risk_Cluster']==risk]
        ax.scatter(sub[xf],sub[yf],c=color,s=25,alpha=0.55,label=f'{risk} (n={len(sub)})')
    # Plot centroids
    for c in range(3):
        risk = risk_map[c]
        ax.scatter(profiles.loc[c,xf],profiles.loc[c,yf],
                   c=risk_colors[risk],s=250,marker='*',edgecolors='black',linewidth=1.5,zorder=10)
        ax.annotate(risk,(profiles.loc[c,xf],profiles.loc[c,yf]),
                    textcoords='offset points',xytext=(6,6),fontsize=8,
                    fontweight='bold',color=risk_colors[risk])
    ax.set_xlabel(xf,fontsize=10); ax.set_ylabel(yf,fontsize=10)
    ax.legend(fontsize=8); ax.grid(True,alpha=0.3)

axes[0].set_title('SST vs Species Cluster Separation',fontweight='bold')
axes[1].set_title('pH vs Species Cluster Separation',fontweight='bold')
axes[2].set_title('SST vs pH Cluster Separation',fontweight='bold')
plt.suptitle('June 18, 2026 — K-Means Risk Zone Segmentation (k=3) Critical / At-Risk / Stable — North Indian Ocean',
             fontsize=13,fontweight='bold')
plt.tight_layout()
plt.savefig(DIRS['visualizations']+'/Week5_Day4_kmeans_clusters.png',dpi=150,bbox_inches='tight')
plt.show(); print('K-Means cluster plot saved ')

In [ ]:
import datetime
cluster_df.to_csv(DIRS['processed']+'/cluster_results.csv',index=False)
model_meta = {'model':'KMeans','k':3,'inertia':round(inertia,4),'n_iter':int(n_iter),
    'random_state':42,'risk_map':{int(k):v for k,v in risk_map.items()},
    'centroids_original_scale':{int(c):{f:round(float(profiles.loc[c,f]),4) for f in FEATURES} for c in range(3)},
    'cluster_sizes':{risk_map[c]:int(counts[c]) for c in range(3)},
    'generated':datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}
with open(DIRS['metadata']+'/kmeans_model_meta.json','w') as f: json.dump(model_meta,f,indent=2)
print('cluster_results.csv + kmeans_model_meta.json saved ')
print()
print('CLUSTER SUMMARY:')
print('   Stable  : C0 — 147 records — SST 27.052°C, pH 8.09715, Species 140.67')
print('   Critical: C1 — 101 records — SST 30.424°C, pH 7.99705, Species  95.68')
print('   At-Risk : C2 — 252 records — SST 28.647°C, pH 8.04346, Species 118.63')